In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import warnings
warnings.filterwarnings('ignore')

# ── Load credentials from .env file ─────────────────
load_dotenv(r"C:\Users\Administrator\Desktop\kenya_road_safety_project\.env")

# ── Database connection ──────────────────────────────
connection_url = URL.create(
    drivername = "postgresql+psycopg2",
    username   = os.getenv("DB_USER"),
    password   = os.getenv("DB_PASSWORD"),
    host       = os.getenv("DB_HOST"),
    port       = int(os.getenv("DB_PORT")),
    database   = os.getenv("DB_NAME")
)
engine = create_engine(connection_url)

# ── File paths ───────────────────────────────────────
BASE_DIR   = r"C:\Users\Administrator\Desktop\kenya_road_safety_project"
RAW_DATA   = os.path.join(BASE_DIR, "data", "raw", "accidents-database-.csv")
CLEAN_OUT  = os.path.join(BASE_DIR, "data", "cleaned", "accidents_clean.csv")

# ── Test connection ──────────────────────────────────
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    print("Connected to PostgreSQL successfully")

print("All imports loaded successfully")


Connected to PostgreSQL successfully
All imports loaded successfully


In [2]:

pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Administrator\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# Load raw CSV
df = pd.read_csv(RAW_DATA)

print("Raw data shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

Raw data shape: (1119, 15)

Columns: ['TIME 24 HOURS', 'BASE/SUB BASE', 'COUNTY', 'ROAD', 'PLACE', 'MV INVOLVED', 'BRIEF ACCIDENT DETAILS', 'NAME OF VICTIM', 'GENDER', 'AGE', 'CAUSE CODE', 'VICTIM', 'NO.', 'Date DD/MM/YYYY', 'Unnamed: 14']

Missing values:
TIME 24 HOURS                5
BASE/SUB BASE                2
COUNTY                       3
ROAD                         3
PLACE                        6
MV INVOLVED                  0
BRIEF ACCIDENT DETAILS       1
NAME OF VICTIM               1
GENDER                       0
AGE                          1
CAUSE CODE                  28
VICTIM                       0
NO.                          4
Date DD/MM/YYYY              0
Unnamed: 14               1117
dtype: int64


In [4]:
print(df.columns.tolist())

['TIME 24 HOURS', 'BASE/SUB BASE', 'COUNTY', 'ROAD', 'PLACE', 'MV INVOLVED', 'BRIEF ACCIDENT DETAILS', 'NAME OF VICTIM', 'GENDER', 'AGE', 'CAUSE CODE', 'VICTIM', 'NO.', 'Date DD/MM/YYYY', 'Unnamed: 14']


In [5]:
df.dtypes

TIME 24 HOURS             object
BASE/SUB BASE             object
COUNTY                    object
ROAD                      object
PLACE                     object
MV INVOLVED               object
BRIEF ACCIDENT DETAILS    object
NAME OF VICTIM            object
GENDER                    object
AGE                       object
CAUSE CODE                object
VICTIM                    object
NO.                       object
Date DD/MM/YYYY           object
Unnamed: 14               object
dtype: object

In [6]:
# Drop the completely empty last column
df = df.drop(columns=['Unnamed: 14'], errors='ignore')

# Rename all columns to clean lowercase names
df.columns = [
    'time', 'base', 'county', 'road', 'place',
    'mv_involved', 'accident_details', 'victim_name',
    'gender', 'age', 'cause_code', 'victim_type',
    'num_victims', 'date'
]

print("Columns after renaming:")
print(df.columns.tolist())
print("\nShape:", df.shape)

Columns after renaming:
['time', 'base', 'county', 'road', 'place', 'mv_involved', 'accident_details', 'victim_name', 'gender', 'age', 'cause_code', 'victim_type', 'num_victims', 'date']

Shape: (1119, 14)


In [7]:
# The date column has mixed formats like 06/25/16 and 6/24
# We force MM/DD/YY format

df['date'] = pd.to_datetime(df['date'], errors='coerce')

print("Date column after conversion:")
print(df['date'].dtype)
print("\nSample dates:")
print(df['date'].head(10))
print("\nNull dates:", df['date'].isnull().sum())

Date column after conversion:
datetime64[ns]

Sample dates:
0   2016-06-25
1   2016-06-25
2   2016-06-25
3   2016-06-25
4   2016-06-25
5   2016-06-25
6   2016-06-25
7          NaT
8          NaT
9          NaT
Name: date, dtype: datetime64[ns]

Null dates: 768


In [8]:
import numpy as np

# Step 1 — Strip HRS suffix to recover valid times
df['time'] = df['time'].astype(str).str.replace(
    'HRS', '', regex=False
).str.strip()

# Step 2 — Replace known bad text values with NaN
df['time'] = df['time'].replace({
    'NAROK'        : np.nan,
    'TIME 24 HOURS': np.nan,
    'UNKNOWN'      : np.nan,
    'UNKNOWN TIME' : np.nan,
    'nan'          : np.nan,
})

# Step 3 — Convert to numeric
df['time'] = pd.to_numeric(df['time'], errors='coerce')

# Step 4 — Extract hour
def extract_hour(t):
    if pd.isnull(t):
        return np.nan
    t_str = str(int(t)).zfill(4)
    return int(t_str[:2])

df['hour'] = df['time'].apply(extract_hour)

# Step 5 — Create time of day
def time_of_day(h):
    if pd.isnull(h):     return 'Unknown'
    if 6  <= h < 12:     return 'Morning'
    if 12 <= h < 17:     return 'Afternoon'
    if 17 <= h < 21:     return 'Evening'
    return 'Night'

df['time_of_day'] = df['hour'].apply(time_of_day)

print("Time of day distribution:")
print(df['time_of_day'].value_counts())
print("\nUnknown times:", (df['time_of_day'] == 'Unknown').sum())

Time of day distribution:
time_of_day
Evening      341
Night        292
Afternoon    231
Morning      230
Unknown       25
Name: count, dtype: int64

Unknown times: 25


In [9]:
# Standardize to uppercase
df['county'] = df['county'].str.strip().str.upper()

# Fix known misspellings and map sub-counties to parent counties
county_corrections = {
    'NAIROBI CITY' : 'NAIROBI',
    'MURANGA'      : "MURANG'A",
    'MURANG A'     : "MURANG'A",
    'THARAKA'      : 'THARAKA NITHI',
    'TAITA'        : 'TAITA TAVETA',
    'KERICHO '     : 'KERICHO',
    'NAROK '       : 'NAROK',
    'TIGANIA'      : 'MERU',
    'ONGATA RONGAI': 'KAJIADO',
    'KITENGELA'    : 'KAJIADO',
    'THIKA'        : 'KIAMBU',
    'MARIAKANI'    : 'KILIFI',
    'VOI'          : 'TAITA TAVETA',
}
df['county'] = df['county'].replace(county_corrections)

# Define all 47 valid Kenya counties
valid_counties = [
    'MOMBASA', 'KWALE', 'KILIFI', 'TANA RIVER', 'LAMU',
    'TAITA TAVETA', 'GARISSA', 'WAJIR', 'MANDERA', 'MARSABIT',
    'ISIOLO', 'MERU', 'THARAKA NITHI', 'EMBU', 'KITUI',
    'MACHAKOS', 'MAKUENI', 'NYANDARUA', 'NYERI', 'KIRINYAGA',
    "MURANG'A", 'KIAMBU', 'TURKANA', 'WEST POKOT', 'SAMBURU',
    'TRANS NZOIA', 'UASIN GISHU', 'ELGEYO MARAKWET', 'NANDI',
    'BARINGO', 'LAIKIPIA', 'NAKURU', 'NAROK', 'KAJIADO',
    'KERICHO', 'BOMET', 'KAKAMEGA', 'VIHIGA', 'BUNGOMA',
    'BUSIA', 'SIAYA', 'KISUMU', 'HOMA BAY', 'MIGORI',
    'KISII', 'NYAMIRA', 'NAIROBI'
]

# Replace anything not a valid county with NaN
df['county'] = df['county'].apply(
    lambda x: x if x in valid_counties else np.nan
)

print("Valid county records :", df['county'].notna().sum())
print("Invalid/null counties:", df['county'].isnull().sum())
print("\nUnique counties found:")
print(sorted(df['county'].dropna().unique()))

Valid county records : 1095
Invalid/null counties: 24

Unique counties found:
['BARINGO', 'BOMET', 'BUNGOMA', 'BUSIA', 'ELGEYO MARAKWET', 'EMBU', 'GARISSA', 'HOMA BAY', 'ISIOLO', 'KAJIADO', 'KAKAMEGA', 'KERICHO', 'KIAMBU', 'KILIFI', 'KIRINYAGA', 'KISII', 'KISUMU', 'KITUI', 'KWALE', 'LAIKIPIA', 'MACHAKOS', 'MAKUENI', 'MANDERA', 'MARSABIT', 'MERU', 'MIGORI', 'MOMBASA', "MURANG'A", 'NAIROBI', 'NAKURU', 'NANDI', 'NAROK', 'NYAMIRA', 'NYANDARUA', 'NYERI', 'SIAYA', 'TAITA TAVETA', 'THARAKA NITHI', 'TRANS NZOIA', 'UASIN GISHU', 'VIHIGA', 'WEST POKOT']


In [10]:
def clean_age_group(age):
    age_str = str(age).strip().upper()

    if age_str in ['NAN', '', 'UNKNOWN']:
        return 'Unknown'
    if age_str == 'J':
        return 'Juvenile'
    if age_str == 'A':
        return 'Adult'
    if '&' in age_str or 'AND' in age_str:
        return 'Multiple'

    # Handle month-based ages — clearly infants = Juvenile
    if 'MONTH' in age_str or 'MTS' in age_str or 'MTH' in age_str:
        return 'Juvenile'

    # Handle comma-separated ages like '46,47,34'
    if ',' in age_str:
        return 'Multiple'

    try:
        a = int(float(age_str))
        if a < 18:  return 'Juvenile'
        if a < 35:  return 'Young Adult'
        if a < 60:  return 'Middle Aged'
        return 'Elderly'
    except:
        return 'Unknown'

df['age_group'] = df['age'].apply(clean_age_group)

print("Age group distribution:")
print(df['age_group'].value_counts())



Age group distribution:
age_group
Adult          545
Young Adult    220
Middle Aged    182
Juvenile        96
Elderly         44
Multiple        27
Unknown          5
Name: count, dtype: int64


In [11]:
def clean_victim(v):
    v = str(v).strip().upper()

    # Handle invalid/header values first
    if v in ['VICTIM', 'NAN', '', 'UNKNOWN', 'RE']:
        return 'Unknown'

    if any(x in v for x in ['M/CYCLIST', 'M/CY', 'MOTOR CYC',
                              'MOTORCYCL', 'M/CYLIST', 'RIDER']):
        return 'Motorcyclist'
    if any(x in v for x in ['PEDESTRIAN', 'PEDST',
                              'PEDESTRAIN', 'PEDSTRIAN']):
        return 'Pedestrian'
    if 'DRIVER' in v:
        return 'Driver'
    if any(x in v for x in ['PASSENGER', 'PASSENG',
                              'PASSANGER', 'PASSEGER']):
        return 'Passenger'
    if any(x in v for x in ['PEDAL CYC', 'P/CYCLIST',
                              'CYCLIST']):
        return 'Cyclist'
    return 'Other'

df['victim_category'] = df['victim_type'].apply(clean_victim)

print("Victim category distribution:")
print(df['victim_category'].value_counts())

#'VICTIM'  → caught at top → Unknown  (was Other)
#'RE'      → caught at top → Unknown  (was Other)
#''        → caught at top → Unknown  (was Other)

Victim category distribution:
victim_category
Pedestrian      455
Motorcyclist    250
Passenger       246
Driver          135
Cyclist          29
Other             2
Unknown           2
Name: count, dtype: int64


In [12]:
# Clean num_victims column
df['num_victims'] = pd.to_numeric(
    df['num_victims'], errors='coerce'
).fillna(1).astype(int)

# Create is_fatal target variable from accident details
fatal_keywords = [
    'DIED', 'DEAD', 'FATAL', 'KILLED',
    'DEATH', 'DOA', 'SUCCUMBED'
]

df['is_fatal'] = df['accident_details'].str.upper().apply(
    lambda x: 1 if any(
        word in str(x) for word in fatal_keywords
    ) else 0
)

print("Num victims distribution:")
print(df['num_victims'].value_counts().head())

print("\nFatality breakdown:")
print(df['is_fatal'].value_counts())
print(f"\nFatal rate: {df['is_fatal'].mean()*100:.1f}%")

Num victims distribution:
num_victims
1    1025
2      60
3      15
5       5
4       5
Name: count, dtype: int64

Fatality breakdown:
is_fatal
0    1079
1      40
Name: count, dtype: int64

Fatal rate: 3.6%


In [13]:
def clean_gender(g):
    g = str(g).strip().upper()

    # Handle invalid header/blank values
    if g in ['NAN', '', 'UNKNOWN', 'GENDER', '1']:
        return 'Unknown'

    # Clean single values first
    if g in ['M', 'MALE']:
        return 'Male'
    if g in ['F', 'FEMALE']:
        return 'Female'

    # M/A means Male/Adult — not multiple genders
    if g == 'M/A':
        return 'Male'

    # Handle juvenile combinations
    if g in ['F/J', 'M/J']:
        return 'Multiple'

    # Handle infant combinations like 4F & INF
    if 'INF' in g:
        return 'Multiple'

    # Handle all other M + F combinations
    if 'M' in g and 'F' in g:
        return 'Multiple'

    return 'Unknown'

df['gender_clean'] = df['gender'].apply(clean_gender)

print("Clean gender distribution:")
print(df['gender_clean'].value_counts())


Clean gender distribution:
gender_clean
Male        942
Female      142
Multiple     28
Unknown       7
Name: count, dtype: int64


In [14]:
print("="*50)
print("CLEANED DATASET SUMMARY")
print("="*50)
print(f"Total records     : {len(df)}")
print(f"Total columns     : {len(df.columns)}")
print(f"\nColumns:")
for col in df.columns:
    nulls = df[col].isnull().sum()
    print(f"  {col:<20} nulls: {nulls}")

print(f"\nFatal accidents   : {df['is_fatal'].sum()}")
print(f"Non-fatal         : {(df['is_fatal']==0).sum()}")
print(f"Unique counties   : {df['county'].nunique()}")
print(f"Unique roads      : {df['road'].nunique()}")
print(f"Date range        : {df['date'].min()} to {df['date'].max()}")

CLEANED DATASET SUMMARY
Total records     : 1119
Total columns     : 20

Columns:
  time                 nulls: 25
  base                 nulls: 2
  county               nulls: 24
  road                 nulls: 3
  place                nulls: 6
  mv_involved          nulls: 0
  accident_details     nulls: 1
  victim_name          nulls: 1
  gender               nulls: 0
  age                  nulls: 1
  cause_code           nulls: 28
  victim_type          nulls: 0
  num_victims          nulls: 0
  date                 nulls: 768
  hour                 nulls: 25
  time_of_day          nulls: 0
  age_group            nulls: 0
  victim_category      nulls: 0
  is_fatal             nulls: 0
  gender_clean         nulls: 0

Fatal accidents   : 40
Non-fatal         : 1079
Unique counties   : 42
Unique roads      : 715
Date range        : 2016-04-13 00:00:00 to 2017-10-31 00:00:00


In [15]:
# Save to cleaned folder
df.to_csv(CLEAN_OUT, index=False)
print(f"Cleaned data saved to:\n{CLEAN_OUT}")

# Also load into PostgreSQL
df.to_sql(
    'accidents_clean',
    engine,
    if_exists='replace',
    index=False,
    chunksize=500
)

# Verify
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM accidents_clean")
    )
    print(f"\nRecords in PostgreSQL: {result.fetchone()[0]}")

print("\nNotebook 2 complete!")


Cleaned data saved to:
C:\Users\Administrator\Desktop\kenya_road_safety_project\data\cleaned\accidents_clean.csv

Records in PostgreSQL: 1119

Notebook 2 complete!
